<a href="https://colab.research.google.com/github/towardsai/ai-tutor-rag-system/blob/main/notebooks/Structured_Outputs_Project_Ticket_Triage.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project 1: Ticket Triage with Confidence-Based Routing

Companion notebook for the lesson **Applied Structured Outputs: Three Mini Projects**.

We build an automated first pass over a support queue: messy, free-form tickets go in, and validated, routable records come out. A confidence threshold decides which results the pipeline acts on autonomously and which go to a human.

**What you will build:**

1. A Pydantic schema with `Literal` enums, numeric constraints, and field descriptions that steer the model.
2. A `triage_ticket()` function with a confidence threshold enforced in code, as policy.
3. A batch loop that separates successes from failures instead of crashing.
4. Deterministic priority overrides layered on top of the model output.

## Install Packages and Set Up the Provider

Pick your provider by setting `PROVIDER` below. Gemini is the course default and its free tier covers this notebook.

> **Colab users:** store your API key in **Secrets** (the key icon in the left sidebar) under the name shown for your provider (`GOOGLE_API_KEY`, `OPENAI_API_KEY`, or `ANTHROPIC_API_KEY`). The cell falls back to an interactive prompt if no secret is found.

In [1]:
# Shared install profile for the structured outputs project notebooks (pin set checked August 13, 2026)
!pip install -q google-genai==2.18.0 openai==3.0.0 anthropic==0.122.0 pydantic==2.13.4 pandas==3.0.5 tqdm==4.70.0

In [2]:
import os
import getpass

PROVIDER = "gemini"  # "gemini" | "openai" | "anthropic"

KEY_ENV = {
    "gemini": "GOOGLE_API_KEY",
    "openai": "OPENAI_API_KEY",
    "anthropic": "ANTHROPIC_API_KEY",
}
env_var = KEY_ENV[PROVIDER]

# Option 1: Colab Secrets (recommended)
try:
    from google.colab import userdata
    os.environ[env_var] = userdata.get(env_var)
except Exception:
    pass

# Option 2: interactive prompt (fallback)
if not os.getenv(env_var):
    os.environ[env_var] = getpass.getpass(f"Enter {env_var}: ")

print(f"[OK] {env_var} is set")

[OK] GOOGLE_API_KEY is set


### The `extract()` Helper

All project code goes through one function, `extract(prompt, schema)`. It sends a prompt and returns a validated Pydantic object, using the native structured output API of whichever provider you selected. We use each provider's small, fast model: extraction is high-volume, low-difficulty work, and the flagship models cost several times more per token while adding little on tasks this constrained. (Model IDs current as of August 13, 2026; swap in the provider's latest small model if these have been superseded.)

In [3]:
from pydantic import BaseModel

MODELS = {
    "gemini": "gemini-3.5-flash-lite",
    "openai": "gpt-5.6-luna",
    "anthropic": "claude-haiku-4-5",
}

if PROVIDER == "gemini":
    from google import genai
    client = genai.Client()
elif PROVIDER == "openai":
    from openai import OpenAI
    client = OpenAI()
elif PROVIDER == "anthropic":
    import anthropic
    client = anthropic.Anthropic()

def extract(prompt: str, schema: type[BaseModel], system: str | None = None,
            model: str | None = None) -> BaseModel:
    """Send a prompt and return a validated instance of `schema`."""
    if PROVIDER == "gemini":
        response = client.models.generate_content(
            model=model or MODELS["gemini"],
            contents=prompt,
            config={
                "system_instruction": system,
                "response_mime_type": "application/json",
                "response_schema": schema,
            },
        )
        if response.parsed is None:
            raise ValueError("Model returned no parseable output")
        return response.parsed
    if PROVIDER == "openai":
        response = client.responses.parse(
            model=model or MODELS["openai"],
            instructions=system,
            input=prompt,
            text_format=schema,
            reasoning={"effort": "none"},
        )
        return response.output_parsed
    if PROVIDER == "anthropic":
        response = client.messages.parse(
            model=model or MODELS["anthropic"],
            max_tokens=2048,
            **({"system": system} if system else {}),
            messages=[{"role": "user", "content": prompt}],
            output_format=schema,
        )
        return response.parsed_output
    raise ValueError(f"Unknown provider: {PROVIDER}")

print(f"[OK] extract() ready, provider={PROVIDER}, model={MODELS[PROVIDER]}")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


[OK] extract() ready, provider=gemini, model=gemini-3.5-flash-lite


## The Sample Ticket Queue

Suppose you run the support desk for an online learning platform. Tickets arrive from a support portal, from email, and from Discord. Here is a small synthetic queue:

In [4]:
tickets = [
    {"id": "TCK-1042", "channel": "portal",
     "text": "Video player on Lesson 12 stuck on 'Loading' for my whole cohort since 9am. "
             "Refreshing does not help. We have a live workshop at noon, please help ASAP."},
    {"id": "TCK-1043", "channel": "email",
     "text": "Need access: please add me to the 'beta-graders' group for the certification "
             "program. My instructor approved it. Username: jdoe."},
    {"id": "TCK-1044", "channel": "discord",
     "text": "FYI: quiz pages feel slower this week. The results dashboard shows grading lag "
             "of ~2h since Monday. No errors, but the trend is concerning."},
    {"id": "TCK-1045", "channel": "portal",
     "text": "Certificate download returns a 500 error for some students since yesterday's "
             "deploy. Error spike visible in Sentry. Might be bad handling of accented names."},
    {"id": "TCK-1046", "channel": "email",
     "text": "A student asks: 'How do I reset my API key for the course exercises?' They got "
             "a quota error. Can someone send instructions?"},
]

print(f"Loaded {len(tickets)} tickets")
for t in tickets:
    print(f"  [{t['id']}] ({t['channel']}) {t['text'][:55]}...")

Loaded 5 tickets
  [TCK-1042] (portal) Video player on Lesson 12 stuck on 'Loading' for my who...
  [TCK-1043] (email) Need access: please add me to the 'beta-graders' group ...
  [TCK-1044] (discord) FYI: quiz pages feel slower this week. The results dash...
  [TCK-1045] (portal) Certificate download returns a 500 error for some stude...
  [TCK-1046] (email) A student asks: 'How do I reset my API key for the cour...


## Step 1: The Triage Schema

The schema is where most of the design work happens:

- `Literal` restricts categorical fields so the model cannot invent a category or priority.
- `Field(ge=..., le=...)` constrains `confidence` to a valid range.
- Every field description is serialized into the JSON schema the model receives, so descriptions act as targeted, per-field instructions.

Note the judgment call on `routing_team`: it stays a plain string with the options listed in its description, because team names change more often than code deploys. Fields that downstream code branches on should be hard enums. Fields a human reads can stay soft.

In [5]:
from pydantic import Field
from typing import Literal

Category = Literal["incident", "access_request", "question", "performance", "bug", "other"]
Priority = Literal["P0", "P1", "P2", "P3"]

class TriageResult(BaseModel):
    """Structured output schema for ticket triage."""

    category: Category = Field(description="High-level ticket type classification")
    priority: Priority = Field(
        description="Urgency and impact. P0 = platform-wide outage, P3 = low priority"
    )
    affected_service: str = Field(description="The system or feature impacted, or 'unknown'")
    summary: str = Field(description="1-2 sentence summary of the issue")
    next_action: str = Field(description="Concrete next step")
    routing_team: str = Field(
        description="Suggested owner team: Platform, Content, Billing, Community, or Infra"
    )
    needs_human_review: bool = Field(
        description="True if a human should confirm before acting"
    )
    confidence: float = Field(ge=0.0, le=1.0, description="Self-reported confidence, 0-1")

print("[OK] TriageResult schema defined")

[OK] TriageResult schema defined


## Step 2: The Triage Function

The function wraps `extract()` with a task prompt and one business rule: low confidence always routes to a human.

Why enforce the threshold in code when the prompt already asks the model to set `needs_human_review`? Because the threshold is policy, and policy belongs where you can change it without re-testing a prompt. The model's own flag captures qualitative doubt. The code rule guarantees the quantitative floor holds even when the model forgets its instructions.

One caveat: self-reported confidence is a ranking signal, not a calibrated probability. A model saying `0.9` is not right 90% of the time. Tune the threshold against a hand-labeled sample (see the exercises).

In [6]:
SYSTEM = """You are the triage assistant for an online learning platform's support desk.
Classify the ticket and extract the requested fields.
Be conservative. If unsure, lower confidence and set needs_human_review to true.
Use P0 only when the platform or a whole course is unusable."""

CONFIDENCE_THRESHOLD = 0.6

def triage_ticket(ticket_text: str) -> TriageResult:
    prompt = f"{SYSTEM}\n\nTicket:\n{ticket_text}"
    result = extract(prompt, TriageResult)

    # Business rule: low confidence always routes to a human
    if result.confidence < CONFIDENCE_THRESHOLD:
        result.needs_human_review = True

    return result

print("[OK] triage_ticket() defined")

[OK] triage_ticket() defined


In [7]:
# Test on a single ticket
result = triage_ticket(tickets[0]["text"])

print(f"Category:     {result.category}")
print(f"Priority:     {result.priority}")
print(f"Service:      {result.affected_service}")
print(f"Summary:      {result.summary}")
print(f"Team:         {result.routing_team}")
print(f"Next action:  {result.next_action}")
print(f"Review:       {result.needs_human_review}  (confidence {result.confidence:.2f})")

Category:     incident
Priority:     P1
Service:      Video player
Summary:      Video player on Lesson 12 is stuck on loading for an entire cohort, impeding preparation for an upcoming noon workshop.
Team:         Infra
Next action:  Investigate Lesson 12 video player streaming server status and notify the content delivery engineering team.
Review:       True  (confidence 0.85)


## Step 3: Triaging the Queue

Real queues have hundreds of tickets, and one malformed ticket must not kill the run. The loop separates successes from failures and keeps going. In production, the failures list gets retried or routed to the same human queue as the low-confidence results.

In [8]:
import time
from tqdm import tqdm

results, failures = [], []

for t in tqdm(tickets, desc="Triaging"):
    try:
        r = triage_ticket(t["text"])
        results.append({"id": t["id"], "channel": t["channel"], **r.model_dump()})
    except Exception as e:
        failures.append({"id": t["id"], "error": str(e)})
    time.sleep(0.2)  # stay under free-tier rate limits

print(f"{len(results)} triaged, {len(failures)} failed")

Triaging:   0%|          | 0/5 [00:00<?, ?it/s]

Triaging:  20%|██        | 1/5 [00:01<00:04,  1.17s/it]

Triaging:  40%|████      | 2/5 [00:02<00:03,  1.02s/it]

Triaging:  60%|██████    | 3/5 [00:03<00:02,  1.06s/it]

Triaging:  80%|████████  | 4/5 [00:04<00:01,  1.11s/it]

Triaging: 100%|██████████| 5/5 [00:05<00:00,  1.12s/it]

Triaging: 100%|██████████| 5/5 [00:05<00:00,  1.10s/it]

5 triaged, 0 failed


In [9]:
import pandas as pd

df = pd.DataFrame(results)
print(df[["id", "category", "priority", "routing_team", "needs_human_review", "confidence"]]
      .to_string(index=False))

      id       category priority routing_team  needs_human_review  confidence
TCK-1042       incident       P1        Infra                True        0.85
TCK-1043 access_request       P3     Platform                True        0.95
TCK-1044    performance       P2        Infra               False        0.90
TCK-1045            bug       P2     Platform               False        0.95
TCK-1046       question       P3     Platform               False        0.95


## Step 4: Rule-Based Priority Overrides

Some routing decisions should never depend on a model at all. If a ticket mentions a 500 error or an outage, we want P0 treatment regardless of what the model concluded. Deterministic post-processing gives you that guarantee, and it is auditable in a way a prompt never is.

The trade-off is that crude rules misfire: the substring `"500"` also matches a ticket about "1500 students". In practice, you refine these rules from real false positives.

In [10]:
def apply_priority_overrides(row: dict) -> dict:
    row = row.copy()
    text = next(t["text"] for t in tickets if t["id"] == row["id"]).lower()

    # 1. Hard keyword rule: outage language forces P0
    if "500" in text or "outage" in text or "down" in text:
        row["priority"] = "P0"

    # 2. Incidents err on the urgent side: bump one level
    bump = {"P3": "P2", "P2": "P1", "P1": "P0"}
    if row["category"] == "incident" and row["priority"] != "P0":
        row["priority"] = bump[row["priority"]]

    return row

for r in results:
    updated = apply_priority_overrides(r)
    marker = "-> " + updated["priority"] if updated["priority"] != r["priority"] else "(no change)"
    print(f"  [{r['id']}] {r['priority']} {marker}")

  [TCK-1042] P1 -> P0
  [TCK-1043] P3 (no change)
  [TCK-1044] P2 (no change)
  [TCK-1045] P2 -> P0
  [TCK-1046] P3 (no change)


## Exercises

1. **Calibrate the threshold.** Hand-label 20 tickets of your own (category + priority). Run the triage function and compare accuracy on high-confidence vs low-confidence outputs. Is `0.6` the right threshold for this model?
2. **Multi-label tickets.** Some tickets are both a `bug` and a `question`. Change `category` to `List[Category]` and update the downstream code. What breaks?
3. **Few-shot examples.** Add two solved example tickets to the prompt. Does agreement with your hand labels improve?
4. **Hierarchical routing.** Add a second extraction step that routes within the chosen team (for example, Platform > video, Platform > auth).

**Reflection questions:**

- How would you handle tickets written in other languages?
- What should happen to a ticket that contains personal data (an email address, a payment reference) before it reaches the API? Project 2 in this lesson gives one answer.